# 22 — Aspect visuals (year slices)

Charts over the Claude silver labels for year slices (2018–2024, en + vi combined):

| # | Chart | Style |
|---|-------|-------|
| 3 | `key_aspect` share over years (5 lines) | line graph |
| 3b | aspect share by language over years | line graph × 2 panels |
| 4 | top-5 `sub_aspect` per `key_aspect` over years (damped support) | line graph × 5 panels |
| 4b | top-5 `sub_aspect` per `key_aspect` over years (weighted volume) | line graph × 5 panels |
| 5 | **COVID recovery** — sub_aspect share change pre vs post 2021 | diverging bar |
| 5b | **COVID recovery** — full sub_aspect trajectory heatmap | year × sub_aspect heatmap |

**Two metrics:**
- **Volume** (`weight × n_reviews`): aspect-level shares.
- **Sub_aspect support** (`Σ weight × log(1 + n_reviews)`): dampened; requires ≥ 2 topics.

**Prerequisite:** run `llm_label_submit.py` + `llm_label_retrieve.py` for year runs first.  
**Coast band visuals → see `18_aspect_visuals.ipynb`.**

In [ ]:
import sys, json
from pathlib import Path
sys.path.append("../src")

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import llm_label as ll

IMG_DIR = Path("../img")
IMG_DIR.mkdir(exist_ok=True)

ASPECT_ORDER = ["facility", "amenity", "service", "experience", "loyalty"]
ASPECT_COLORS = {
    "facility":   "#4878a8",
    "amenity":    "#6a9f58",
    "service":    "#e49444",
    "experience": "#d1605e",
    "loyalty":    "#a87ca8",
}
TITLE_KW = dict(fontsize=16, fontweight="bold", color="#36648B", loc="left")

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 14, "axes.labelsize": 12,
                     "xtick.labelsize": 11, "ytick.labelsize": 11})

## Load labeled data

In [ ]:
con = duckdb.connect(str(ll.DB_PATH), read_only=True)

df = con.execute("""
    SELECT rt.run_id, rt.topic_id, ta.key_aspect, ta.weight, ta.sentiment,
           ta.sub_aspects, COUNT(*) AS n_reviews
    FROM REVIEW_TOPICS rt
    JOIN TOPIC_ASPECTS ta
      ON rt.run_id = ta.run_id AND rt.topic_id = ta.topic_id
    WHERE rt.topic_id != -1 AND ta.key_aspect != 'other'
      AND rt.run_id LIKE 'year_%'
    GROUP BY ALL
""").df()
con.close()

if df.empty:
    raise RuntimeError("No year runs labeled yet — run llm_label_submit.py for year runs first.")

df["weighted"] = df["weight"] * df["n_reviews"]
df["language"] = df["run_id"].str[-2:]

year = df.copy()
year["year"] = year["run_id"].str.extract(r"year_(\d{4})")[0].astype(int)

print(f"labeled aspect rows: {len(year):,}")
print("labeled runs:", sorted(year.run_id.unique()))

## Chart 3 — Aspect share over years
5 lines, one per `key_aspect`; y = % of weighted reviews within the year (en + vi combined).

In [ ]:
pivot = (year.groupby(["year", "key_aspect"])["weighted"].sum()
         .unstack("key_aspect").reindex(columns=ASPECT_ORDER).fillna(0))
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).mul(100)

xticks = sorted(year["year"].unique())

fig, ax = plt.subplots(figsize=(12, 6))
for aspect in ASPECT_ORDER:
    ax.plot(pivot_pct.index, pivot_pct[aspect], marker="o", linewidth=2.5,
            color=ASPECT_COLORS[aspect], label=aspect)
    ax.text(pivot_pct.index[-1] + 0.12, pivot_pct[aspect].iloc[-1],
            aspect, color=ASPECT_COLORS[aspect], fontsize=10, fontweight="bold", va="center")

ax.set_xticks(xticks)
ax.set_xlabel("Year")
ax.set_ylabel("Aspect share (%)")
ax.set_title("Aspect share over years (en + vi combined)", **TITLE_KW)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(IMG_DIR / "22_year_aspect_lines.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 3c — Key_aspect share over years with COVID context

Same 5-line trajectory as chart 3, with three visual layers added:
- **Gray band** = COVID window (2020–2021)
- **Dashed line** = pre-COVID average per aspect (2018–2019)
- **Dotted line** = post-COVID average per aspect (2022–2024)

This makes the structural shift visible: lines that end above their dashed baseline grew after COVID; lines below shrank.

In [ ]:
PRE_YEARS  = [2018, 2019]
COVID_YEARS = [2020, 2021]
POST_YEARS = [2022, 2023, 2024]

pivot = (year.groupby(["year", "key_aspect"])["weighted"].sum()
         .unstack("key_aspect").reindex(columns=ASPECT_ORDER).fillna(0))
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).mul(100)

xticks = sorted(year["year"].unique())

# pre / post averages per aspect
pre_avg  = pivot_pct.loc[pivot_pct.index.isin(PRE_YEARS)].mean()
post_avg = pivot_pct.loc[pivot_pct.index.isin(POST_YEARS)].mean()

fig, ax = plt.subplots(figsize=(13, 6.5))

# COVID shaded band
covid_in_data = [y for y in COVID_YEARS if y in xticks]
if covid_in_data:
    ax.axvspan(min(covid_in_data) - 0.5, max(covid_in_data) + 0.5,
               color="steelblue", alpha=0.08, zorder=0, label="_nolegend_")
    ax.text((min(covid_in_data) + max(covid_in_data)) / 2,
            pivot_pct.max().max() * 0.97,
            "COVID\n2020–2021", ha="center", va="top",
            fontsize=9, color="steelblue", style="italic")

# pre / post shaded regions
pre_in_data = [y for y in PRE_YEARS if y in xticks]
post_in_data = [y for y in POST_YEARS if y in xticks]
if pre_in_data:
    ax.axvspan(min(pre_in_data) - 0.5, max(pre_in_data) + 0.5,
               color="#f5a623", alpha=0.06, zorder=0)
    ax.text(min(pre_in_data) - 0.4, pivot_pct.max().max() * 0.97,
            "Pre-COVID", ha="left", va="top", fontsize=8.5,
            color="#c8860a", style="italic")
if post_in_data:
    ax.axvspan(min(post_in_data) - 0.5, max(post_in_data) + 0.5,
               color="#27ae60", alpha=0.06, zorder=0)
    ax.text(max(post_in_data) + 0.4, pivot_pct.max().max() * 0.97,
            "Recovery", ha="right", va="top", fontsize=8.5,
            color="#1e8449", style="italic")

for aspect in ASPECT_ORDER:
    color = ASPECT_COLORS[aspect]
    # main trajectory line
    ax.plot(pivot_pct.index, pivot_pct[aspect], marker="o", linewidth=2.5,
            color=color, zorder=3)
    # right-side label
    ax.text(xticks[-1] + 0.12, pivot_pct[aspect].iloc[-1],
            aspect, color=color, fontsize=10, fontweight="bold", va="center")

    # pre-COVID average — dashed
    if pre_in_data:
        ax.hlines(pre_avg[aspect],
                  min(pre_in_data) - 0.45, max(pre_in_data) + 0.45,
                  colors=color, linewidths=1.4, linestyles="dashed",
                  alpha=0.6, zorder=2)
    # post-COVID average — dotted
    if post_in_data:
        ax.hlines(post_avg[aspect],
                  min(post_in_data) - 0.45, max(post_in_data) + 0.45,
                  colors=color, linewidths=1.4, linestyles="dotted",
                  alpha=0.6, zorder=2)

    # delta arrow between the two averages (drawn at rightmost post year)
    if pre_in_data and post_in_data:
        xa = max(post_in_data) + 0.38
        dy = post_avg[aspect] - pre_avg[aspect]
        if abs(dy) > 0.3:
            ax.annotate("", xy=(xa, post_avg[aspect]),
                        xytext=(xa, pre_avg[aspect]),
                        arrowprops=dict(arrowstyle="->" if dy > 0 else "<-",
                                        color=color, lw=1.2))

ax.set_xticks(xticks)
ax.set_xlabel("Year")
ax.set_ylabel("Aspect share (%)")
ax.set_title("Key_aspect share trajectory with COVID context\n"
             "(dashed = pre-COVID avg · dotted = post-COVID avg · arrows = shift direction)",
             **TITLE_KW)
ax.grid(axis="y", alpha=0.2)

from matplotlib.lines import Line2D
legend_items = [
    Line2D([0], [0], color="gray", linewidth=1.4, linestyle="dashed"),
    Line2D([0], [0], color="gray", linewidth=1.4, linestyle="dotted"),
]
ax.legend(legend_items, ["pre-COVID avg (2018–2019)", "post-COVID avg (2022–2024)"],
          loc="lower left", fontsize=9, frameon=False)

fig.tight_layout()
fig.savefig(IMG_DIR / "22_year_aspect_covid_context.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 3b — Aspect share by language over years
Same as chart 3 but split into English (left) and Vietnamese (right) panels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)

for ax, lang, title in zip(axes, ["en", "vi"], ["English", "Vietnamese"]):
    d = year[year["language"] == lang]
    if d.empty:
        ax.set_title(f"{title} — no data", **TITLE_KW)
        continue
    pv = (d.groupby(["year", "key_aspect"])["weighted"].sum()
          .unstack("key_aspect").reindex(columns=ASPECT_ORDER).fillna(0))
    pv_pct = pv.div(pv.sum(axis=1), axis=0).mul(100)
    for aspect in ASPECT_ORDER:
        ax.plot(pv_pct.index, pv_pct[aspect], marker="o", linewidth=2.2,
                color=ASPECT_COLORS[aspect], label=aspect)
        ax.text(pv_pct.index[-1] + 0.12, pv_pct[aspect].iloc[-1],
                aspect, color=ASPECT_COLORS[aspect], fontsize=9, fontweight="bold", va="center")
    ax.set_xticks(sorted(d["year"].unique()))
    ax.set_xlabel("Year")
    ax.set_title(title, **TITLE_KW)
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Aspect share (%)")
fig.suptitle("Aspect share over years — by language",
             fontsize=16, fontweight="bold", color="#36648B", x=0.01, ha="left")
fig.tight_layout()
fig.savefig(IMG_DIR / "22_year_aspect_by_lang.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 4 — Top-5 sub_aspects per key_aspect over years (damped support)

One panel per aspect; top-5 sub_aspects ranked by `Σ weight × log(1 + n_reviews)`.
Requires ≥ 2 supporting topics per sub_aspect. Generic sub_aspect labels removed.

In [ ]:
GENERIC_SUBS = {
    "facility":   {"facility", "facilities", "room", "cleanliness", "hygiene"},
    "amenity":    {"amenity", "amenities", "location", "place", "area"},
    "service":    {"service", "staff"},
    "experience": {"experience"},
    "loyalty":    {"loyalty"},
}


def explode_subs(d: pd.DataFrame) -> pd.DataFrame:
    s = d.copy()
    s["sub_aspect"] = s["sub_aspects"].apply(json.loads)
    s = s.explode("sub_aspect").dropna(subset=["sub_aspect"])
    s["topic_key"] = s["run_id"] + "/" + s["topic_id"].astype(str)
    s["support"] = s["weight"] * np.log1p(s["n_reviews"])
    return s


def top_subaspects(s: pd.DataFrame, aspect: str, k: int = 5,
                   min_topics: int = 2) -> pd.Index:
    d = s[(s["key_aspect"] == aspect)
          & ~s["sub_aspect"].isin(GENERIC_SUBS.get(aspect, set()))]
    g = d.groupby("sub_aspect").agg(n_topics=("topic_key", "nunique"),
                                    score=("support", "sum"))
    eligible = g[g["n_topics"] >= min_topics]
    if len(eligible) < k:
        eligible = g
    return eligible.sort_values("score", ascending=False).head(k)


ysubs = explode_subs(year)

fig, axes = plt.subplots(len(ASPECT_ORDER), 1,
                         figsize=(12, 3.4 * len(ASPECT_ORDER)), sharex=True)
xticks = sorted(year["year"].unique())

for ax, aspect in zip(axes, ASPECT_ORDER):
    top = top_subaspects(ysubs, aspect)
    if top.empty:
        ax.set_visible(False)
        continue
    d = ysubs[(ysubs["key_aspect"] == aspect) & ysubs["sub_aspect"].isin(top.index)]
    p = (d.groupby(["year", "sub_aspect"])["support"].sum()
         .unstack("sub_aspect").reindex(columns=top.index).fillna(0))
    for i, sub_name in enumerate(top.index):
        ax.plot(p.index, p[sub_name], marker="o", linewidth=2,
                alpha=1 - 0.15 * i, color=ASPECT_COLORS[aspect], label=sub_name)
    ax.set_title(aspect.upper(), fontsize=13, fontweight="bold",
                 color=ASPECT_COLORS[aspect], loc="left")
    ax.grid(axis="y", alpha=0.3)
    ax.set_ylabel("support score")
    ax.legend(fontsize=9, ncol=2, frameon=False)

axes[-1].set_xticks(xticks)
axes[-1].set_xlabel("Year")
fig.suptitle("Top-5 sub_aspects per key_aspect over years (damped support)",
             fontsize=16, fontweight="bold", color="#36648B", x=0.01, ha="left")
fig.tight_layout(rect=[0, 0, 1, 0.99])
fig.savefig(IMG_DIR / "22_year_subaspect_lines.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 4b — Top-5 sub_aspects per key_aspect over years (weighted volume)
Same layout but ranked by raw `weight × n_reviews`. Useful for comparing absolute volume.

In [ ]:
fig, axes = plt.subplots(len(ASPECT_ORDER), 1,
                         figsize=(12, 3.4 * len(ASPECT_ORDER)), sharex=True)
xticks = sorted(year["year"].unique())

for ax, aspect in zip(axes, ASPECT_ORDER):
    d = ysubs[ysubs["key_aspect"] == aspect]
    if d.empty:
        ax.set_visible(False)
        continue
    top5 = d.groupby("sub_aspect")["weighted"].sum().nlargest(5).index
    p = (d[d["sub_aspect"].isin(top5)]
         .groupby(["year", "sub_aspect"])["weighted"].sum()
         .unstack("sub_aspect").reindex(columns=top5).fillna(0))
    for i, sub_name in enumerate(top5):
        ax.plot(p.index, p[sub_name], marker="o", linewidth=2,
                alpha=1 - 0.15 * i, color=ASPECT_COLORS[aspect], label=sub_name)
    ax.set_title(aspect.upper(), fontsize=13, fontweight="bold",
                 color=ASPECT_COLORS[aspect], loc="left")
    ax.grid(axis="y", alpha=0.3)
    ax.set_ylabel("weighted reviews")
    ax.legend(fontsize=9, ncol=2, frameon=False)

axes[-1].set_xticks(xticks)
axes[-1].set_xlabel("Year")
fig.suptitle("Top-5 sub_aspects per key_aspect over years (weighted volume)",
             fontsize=16, fontweight="bold", color="#36648B", x=0.01, ha="left")
fig.tight_layout(rect=[0, 0, 1, 0.99])
fig.savefig(IMG_DIR / "22_year_subaspect_volume.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 5 — COVID recovery: sub_aspect share change pre vs post 2021

Diverging bar chart: one bar per sub_aspect, showing the change in share from the
**pre-COVID period (2018–2019)** to the **recovery period (2022–2024)**.

- **Green (right)** = guests care more about this after COVID
- **Red (left)** = guests care less / mention it less after COVID
- Bars are colored by their dominant `key_aspect`
- Share = sub_aspect support / total support across all sub_aspects for that year group

COVID period (2020–2021) is deliberately excluded from both baselines to isolate the
structural shift rather than the pandemic trough.

In [ ]:
PRE_COVID  = [2018, 2019]        # baseline
POST_COVID = [2022, 2023, 2024]  # recovery

# sub_aspect support per year group (exclude 2020-2021 from both baselines)
def period_shares(years: list[int]) -> pd.Series:
    d = ysubs[ysubs["year"].isin(years)
              & ~ysubs["sub_aspect"].isin(
                  {s for subs in GENERIC_SUBS.values() for s in subs}
              )]
    total = d["support"].sum()
    return d.groupby("sub_aspect")["support"].sum() / total * 100

pre  = period_shares(PRE_COVID)
post = period_shares(POST_COVID)

all_subs = pre.index.union(post.index)
pre  = pre.reindex(all_subs, fill_value=0)
post = post.reindex(all_subs, fill_value=0)
delta_full = post - pre

# top 10 positive + top 10 negative only
top_pos = delta_full[delta_full > 0].nlargest(10)
top_neg = delta_full[delta_full < 0].nsmallest(10)
delta = pd.concat([top_neg, top_pos]).sort_values()

# dominant aspect per sub_aspect (for bar color)
dom_aspect = (ysubs[~ysubs["sub_aspect"].isin(
                  {s for subs in GENERIC_SUBS.values() for s in subs})]
              .groupby(["sub_aspect", "key_aspect"])["support"].sum()
              .reset_index()
              .sort_values("support", ascending=False)
              .drop_duplicates("sub_aspect")
              .set_index("sub_aspect")["key_aspect"])

fig, ax = plt.subplots(figsize=(12, max(6, 0.42 * len(delta) + 2)))

ax.barh(delta.index, delta.values, height=0.72,
        color=[ASPECT_COLORS.get(dom_aspect.get(s, ""), "#aaaaaa") for s in delta.index],
        alpha=0.85, edgecolor="white", linewidth=0.4)

ax.axvline(0, color="black", linewidth=1.2)

# labels: always outside the bar end
PAD = 0.05
for sub, v in zip(delta.index, delta.values):
    if v >= 0:
        ax.text(v + PAD, sub, f"+{v:.2f}pp", va="center", ha="left",
                fontsize=8.5, color="dimgray")
    else:
        ax.text(v - PAD, sub, f"{v:.2f}pp", va="center", ha="right",
                fontsize=8.5, color="dimgray")

# expand xlim so labels aren't clipped
x_margin = delta.abs().max() * 0.35
ax.set_xlim(delta.min() - x_margin, delta.max() + x_margin)

# legend: aspect color patches
handles = [plt.Rectangle((0, 0), 1, 1, color=ASPECT_COLORS[a], alpha=0.85)
           for a in ASPECT_ORDER]
ax.legend(handles, ASPECT_ORDER, loc="lower right", fontsize=9,
          frameon=False, title="dominant aspect")

ax.set_xlabel(f"Percentage-point change in sub_aspect share\n"
              f"(post-COVID {POST_COVID[0]}–{POST_COVID[-1]}  minus  "
              f"pre-COVID {PRE_COVID[0]}–{PRE_COVID[-1]})")
ax.set_title(f"What changed in guest priorities after COVID  "
             f"(top 10 gains + top 10 losses)\n"
             f"colored by dominant aspect",
             **TITLE_KW)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(IMG_DIR / "22_covid_delta_bar.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 5v — COVID recovery: diverging bar (vertical layout)

Same data as chart 5 but with sub_aspects on the x-axis and pp change on the y-axis.
Negative bars hang below the zero line, positive bars rise above it.
Useful for slide/poster layouts where width is constrained.

In [ ]:
# delta and dom_aspect are computed in chart 5's cell — run that first

bar_colors_v = [ASPECT_COLORS.get(dom_aspect.get(s, ""), "#aaaaaa") for s in delta.index]

fig, ax = plt.subplots(figsize=(14, 7))

ax.bar(range(len(delta)), delta.values, width=0.7,
       color=bar_colors_v, alpha=0.85, edgecolor="white", linewidth=0.5)

ax.axhline(0, color="black", linewidth=1.2)

# labels outside bar ends
PAD_V = delta.abs().max() * 0.04
for i, (sub, v) in enumerate(zip(delta.index, delta.values)):
    if v >= 0:
        ax.text(i, v + PAD_V, f"+{v:.2f}pp", ha="center", va="bottom",
                fontsize=8, color="dimgray", rotation=45)
    else:
        ax.text(i, v - PAD_V, f"{v:.2f}pp", ha="center", va="top",
                fontsize=8, color="dimgray", rotation=45)

# expand ylim so rotated labels aren't clipped
y_margin = delta.abs().max() * 0.45
ax.set_ylim(delta.min() - y_margin, delta.max() + y_margin)

ax.set_xticks(range(len(delta)))
ax.set_xticklabels(delta.index, rotation=60, ha="right", fontsize=9)
ax.set_ylabel("Percentage-point change in sub_aspect share")
ax.set_title(f"What changed in guest priorities after COVID  "
             f"(top 10 gains + top 10 losses)\n"
             f"pre-COVID {PRE_COVID[0]}–{PRE_COVID[-1]}  →  "
             f"post-COVID {POST_COVID[0]}–{POST_COVID[-1]}",
             **TITLE_KW)
ax.grid(axis="y", alpha=0.25)

handles = [plt.Rectangle((0, 0), 1, 1, color=ASPECT_COLORS[a], alpha=0.85)
           for a in ASPECT_ORDER]
ax.legend(handles, ASPECT_ORDER, loc="upper left", fontsize=9,
          frameon=False, title="dominant aspect")

fig.tight_layout()
fig.savefig(IMG_DIR / "22_covid_delta_bar_vertical.png", dpi=150, bbox_inches="tight")
plt.show()

## Chart 5b — COVID trajectory heatmap (year × sub_aspect)

Full temporal arc for every sub_aspect: each cell = that sub_aspect's share of total
support in that year. Rows sorted by post-COVID delta (same order as chart 5).
Vertical dashed lines mark the COVID window (2020–2021).

Color scale is **row-normalized** (each sub_aspect scaled to its own max) so slow-moving
sub_aspects don't disappear behind dominant ones.

In [ ]:
all_years = sorted(year["year"].unique())

# sub_aspect × year support matrix (drop generics)
subs_clean = ysubs[~ysubs["sub_aspect"].isin(
    {s for subs in GENERIC_SUBS.values() for s in subs}
)]
mat = (subs_clean.groupby(["sub_aspect", "year"])["support"].sum()
       .unstack("year").reindex(columns=all_years).fillna(0))

# keep only sub_aspects that appear in delta (have pre or post data)
mat = mat.reindex(index=delta.index).fillna(0)

# sort rows by delta (ascending = most-shrunk at top, most-grown at bottom)
mat = mat.loc[delta.sort_values().index]

# row-normalize: each row scaled 0→1 relative to its own max
row_max = mat.max(axis=1).replace(0, 1)
mat_norm = mat.div(row_max, axis=0)

# aspect color strip on the left
aspect_colors_strip = [ASPECT_COLORS.get(dom_aspect.get(s, ""), "#aaaaaa")
                       for s in mat_norm.index]

fig, (ax_strip, ax_hm) = plt.subplots(
    1, 2, figsize=(13, max(7, 0.30 * len(mat_norm) + 2)),
    gridspec_kw={"width_ratios": [0.04, 1]},
)

# color strip
for i, color in enumerate(aspect_colors_strip):
    ax_strip.barh(i, 1, height=0.9, color=color, alpha=0.85)
ax_strip.set_xlim(0, 1)
ax_strip.set_ylim(-0.5, len(mat_norm) - 0.5)
ax_strip.axis("off")

# heatmap
im = ax_hm.imshow(mat_norm.values, aspect="auto", cmap="YlOrRd",
                  vmin=0, vmax=1, origin="lower")

ax_hm.set_xticks(range(len(all_years)))
ax_hm.set_xticklabels(all_years, fontsize=10)
ax_hm.set_yticks(range(len(mat_norm)))
ax_hm.set_yticklabels(mat_norm.index, fontsize=9)

# COVID window
for yr in [2020, 2021]:
    if yr in all_years:
        xi = all_years.index(yr)
        ax_hm.axvline(xi - 0.5, color="steelblue", linewidth=1.5,
                      linestyle="--", alpha=0.8)
        ax_hm.axvline(xi + 0.5, color="steelblue", linewidth=1.5,
                      linestyle="--", alpha=0.8)

# annotate first/last COVID year
for yr, label in [(2020, "COVID\nstart"), (2021, "COVID\nend")]:
    if yr in all_years:
        xi = all_years.index(yr)
        ax_hm.text(xi, len(mat_norm) + 0.3, label, ha="center",
                   fontsize=8, color="steelblue", va="bottom")

plt.colorbar(im, ax=ax_hm, label="Row-normalized support (0 = min, 1 = max)",
             shrink=0.6, pad=0.02)

# aspect legend
handles = [plt.Rectangle((0, 0), 1, 1, color=ASPECT_COLORS[a], alpha=0.85)
           for a in ASPECT_ORDER]
ax_hm.legend(handles, ASPECT_ORDER, loc="upper left",
             bbox_to_anchor=(1.12, 1.0), frameon=False,
             fontsize=9, title="dominant aspect")

ax_hm.set_title("Sub_aspect trajectory — rows sorted by post-COVID growth\n"
                "(blue dashed = COVID window 2020–2021)",
                **TITLE_KW)
fig.tight_layout()
fig.savefig(IMG_DIR / "22_covid_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()